# SW-UDE paso a paso con datos reales

Este cuaderno entrena **solo SW-UDE** sobre la serie de COVID-19 incluida en el repositorio. Reconstruye la trayectoria SIR de referencia, ajusta la red por ventanas temporales, calcula métricas sencillas y muestra las curvas aprendidas. No entrena ni compara contra UDE.

La corrida usa una semilla y los parámetros del experimento 14. Sirve para aprender y examinar una ejecución; no sustituye el análisis multisemilla del artículo.


## Antes de ejecutar

Abre el cuaderno con un kernel de Julia 1.12 y ejecuta las celdas en orden. El proyecto y el Manifest.toml están en la raíz del repositorio; la primera celda activa ese entorno e instala lo que falte. Si no aparece un kernel Julia en Jupyter, instala IJulia en tu entorno global de Julia y registra el kernel antes de abrir el cuaderno.

La configuración por defecto usa 8.000 épocas, igual que el experimento 14, y una semilla. Para comprobar rápidamente el flujo, cambia EPOCHS a 30 en la celda de parámetros; esa corrida reducida es solo una prueba y no se debe interpretar como resultado del artículo.


## 1. Activar el proyecto y cargar la biblioteca
La búsqueda de la raíz permite abrir el cuaderno desde la carpeta del repositorio o desde notebooks/.

In [ ]:
import Pkg

function find_repo_root(start=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "sir_ude_lib.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("No encuentro la raíz del repositorio desde: $start")
        dir = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
Pkg.instantiate()
include(joinpath(REPO_ROOT, "src", "sir_ude_lib.jl"))

println("Proyecto: ", REPO_ROOT)
println("Julia: ", VERSION)

## 2. Cargar los datos y construir la referencia SIR
La serie medida es I(t). S(t) y R(t) se reconstruyen con las condiciones de referencia fijadas en la biblioteca; no son observaciones independientes.

In [ ]:
data_path = joinpath(REPO_ROOT, "data", "historical_data_july_january.csv")
I_raw = cargar_serie_real(data_path)
reference = construir_serie_referencia_fija(I_raw)
SIR = reference.SIR

N = N_REF_FIJO
gamma = GAMMA_REF_FIJO
cfg = make_real_data_config(SIR; N=N, γ=gamma)

println("Observaciones: ", size(SIR, 2))
println("Transiciones: ", cfg.T)
println("Días excluidos antes del anclaje: ", reference.n_dias_descartados)
println("Factor aplicado a la serie: ", round(reference.factor_reescala; digits=4))

In [ ]:
plot(cfg.ts, SIR[2, :];
    label="I observada y reescalada",
    xlabel="Día desde el anclaje", ylabel="Casos",
    title="Serie usada por SW-UDE", linewidth=2.5, color=:black,
    legend=:topright, size=(850, 420))

## 3. Definir la ventana y los parámetros de entrenamiento
Cada ventana de longitud m empieza desde un estado observado. Solo se optimiza la pérdida de entrenamiento; el tramo final queda reservado para validación cronológica.

In [ ]:
const WINDOW = 10
const FRAC_TRAIN = 0.8
const SEED = 1
const HIDDEN = 16
const LAYERS = 2
const LR = 0.005
const EPOCHS = 8000

idx_train, idx_test = split_train_test(cfg.T; frac_train=FRAC_TRAIN)
@assert 1 <= WINDOW <= length(idx_train) "La ventana debe caber en el tramo de entrenamiento"
println("m = $WINDOW días · entrenamiento = $(length(idx_train)) transiciones · validación = $(length(idx_test)) transiciones")

## 4. Entrenar SW-UDE
Se llama directamente a train_windowed_rd. Esta celda no ejecuta el entrenamiento global.

In [ ]:
network, p0, state = build_network(HIDDEN, LAYERS;
    seed=SEED, bias_inicial=Float32(BETA0_REF_FIJO))

@time p_swude, training_loss = train_windowed_rd(cfg, SIR, network, state, p0;
    m=WINDOW, epochs=EPOCHS, lr=LR, idx_entren=idx_train)

println("Épocas completadas: ", length(training_loss))
println("Pérdida final: ", training_loss[end])

## 5. Generar la trayectoria, beta estimada y R efectivo
El rollout comienza desde el estado inicial y luego avanza de forma autónoma. En los datos reales no se conoce la beta verdadera, así que el cuaderno grafica únicamente la beta que aprendió SW-UDE.

In [ ]:
prediction = rollout_rd(cfg, p_swude, state, network)
beta_hat = [beta_theta_rd(cfg, cfg.ts[k], prediction[2, k], p_swude, state, network)
            for k in 1:cfg.T]
reff = r_effective(beta_hat, prediction[1, 1:cfg.T], N, gamma)

train_days = 1:(last(idx_train) + 1)
test_days = (first(idx_test) + 1):(last(idx_test) + 1)
rmse(x, y) = sqrt(mean((x .- y) .^ 2))

metrics = (
    rmse_I_train = rmse(prediction[2, train_days], SIR[2, train_days]),
    rmse_I_validation = rmse(prediction[2, test_days], SIR[2, test_days]),
    beta_min = minimum(beta_hat),
    beta_max = maximum(beta_hat),
    reff_max = maximum(reff),
)
metrics

## 6. Inspeccionar los resultados
En la primera figura, la línea vertical marca el inicio del tramo de validación. Las otras figuras muestran la beta aprendida, R efectivo y la convergencia de la pérdida de SW-UDE.

In [ ]:
split_day = cfg.ts[last(idx_train) + 1]
p_I = plot(cfg.ts, SIR[2, :]; label="Observado", color=:black, linewidth=2.5,
    xlabel="Día desde el anclaje", ylabel="Casos",
    title="SW-UDE: ajuste de I(t)", size=(850, 420))
plot!(p_I, cfg.ts, prediction[2, :]; label="SW-UDE (m=$WINDOW)", color=:steelblue, linewidth=2)
vline!(p_I, [split_day]; label="Inicio de validación", color=:gray, linestyle=:dash)
p_I

In [ ]:
p_beta = plot(cfg.ts[1:cfg.T], beta_hat; label="β̂(t,I) — SW-UDE", color=:darkorange,
    xlabel="Día desde el anclaje", ylabel="β̂", title="Tasa de transmisión aprendida",
    linewidth=2, size=(850, 420))
vline!(p_beta, [split_day]; label="Inicio de validación", color=:gray, linestyle=:dash)
p_beta

In [ ]:
p_reff = plot(cfg.ts[1:cfg.T], reff; label="R_eff — SW-UDE", color=:seagreen,
    xlabel="Día desde el anclaje", ylabel="R_eff(t)", title="Número reproductivo efectivo",
    linewidth=2, size=(850, 420))
hline!(p_reff, [1.0]; label="R_eff = 1", color=:black, linestyle=:dot)
vline!(p_reff, [split_day]; label="Inicio de validación", color=:gray, linestyle=:dash)
p_reff

In [ ]:
plot(eachindex(training_loss), training_loss; label="SW-UDE", color=:purple,
    xlabel="Época", ylabel="Pérdida de entrenamiento", title="Convergencia",
    yscale=:log10, linewidth=2, size=(850, 420))

## 7. Leer las métricas y guardar las figuras
RMSE de validación mide el ajuste en el tramo final reservado. En datos reales, beta no tiene una referencia verdadera; interprétala como una tasa inferida bajo los supuestos de reconstrucción SIR, no como una medición directa.

In [ ]:
output_dir = joinpath(REPO_ROOT, "notebooks", "outputs", "swude_real_data")
mkpath(output_dir)
savefig(p_I, joinpath(output_dir, "swude_infected_cases.png"))
savefig(p_beta, joinpath(output_dir, "swude_beta_estimate.png"))
savefig(p_reff, joinpath(output_dir, "swude_effective_reproduction_number.png"))
savefig(last(current()), joinpath(output_dir, "swude_training_loss.png"))
println("Figuras guardadas en: ", output_dir)

### Qué reportar

- La ventana m, arquitectura, tasa de aprendizaje, número de épocas y semilla.
- RMSE de I(t) en entrenamiento y validación.
- La forma temporal de β̂ y R_eff, recordando que S(t) y R(t) fueron reconstruidos a partir de I(t).
- Este cuaderno ejecuta una sola semilla; para reproducir estadísticas multisemilla, usa los runners del artículo.
